In [2]:
import os
from dotenv import load_dotenv
from google import genai

# Carga la API key guardada en el archivo .env de la raíz
load_dotenv()

# Inicializa el cliente oficial del SDK google-genai
client = genai.Client()

# Llamada de prueba al modelo
response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents="¡Hola desde un Jupyter Notebook! Confirma que la configuración funciona correctamente.",
)

print("Respuesta de Gemini:\n")
print(response.text)

Respuesta de Gemini:

¡Hola! 👋

Te confirmo que la configuración funciona **correctamente**. La conexión entre tu Jupyter Notebook y el modelo está completamente activa y lista para usar.

¿En qué te puedo ayudar hoy? Puedo ayudarte a:
* 🐍 Escribir, optimizar o depurar código Python.
* 📊 Analizar y visualizar datos (Pandas, NumPy, Matplotlib, Seaborn, etc.).
* 🤖 Implementar modelos de Machine Learning o Deep Learning.
* 📝 Redactar texto explicativo en formato Markdown para tu notebook.

¡Dime qué proyecto tienes en mente y empezamos!


In [3]:
prompt_poblacion = (
    "¿Cuál es la población actual de Mallorca? "
    "Dame la cifra total aproximada y un breve desglose por sus municipios principales."
)

response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents=prompt_poblacion,
)

print(response.text)

La población residente en Mallorca se sitúa aproximadamente en torno a los **950.000 y 960.000 habitantes** (según las últimas actualizaciones del Instituto Nacional de Estadística, INE). 

*Nota: Esta cifra se refiere a la **población empadronada/residente**. Durante los meses de verano, la "población flotante" (turistas y trabajadores de temporada) hace que el número real de personas en la isla supere con frecuencia el millón y medio.*

---

### Desglose por sus municipios principales

La población en Mallorca está muy concentrada en la capital y el área metropolitana, mientras que el resto se reparte en municipios de tamaño medio y pequeños pueblos. 

Aquí tienes el desglose de las localidades más pobladas (cifras aproximadas):

1. **Palma (la capital):** 
   * **~425.000 habitantes.**
   * Concentra casi el 45% de toda la población de la isla. Es el centro económico, administrativo y de servicios.

2. **Calvià:** 
   * **~53.000 habitantes.**
   * Situado en el sudoeste, es el prin

In [4]:
from google.genai import types

def get_completion(prompt: str, model: str = "gemini-3.6-flash", temperature: float = 0.0) -> str:
    config = types.GenerateContentConfig(
        temperature=temperature
    )
    response = client.models.generate_content(
        model=model,
        contents=prompt,
        config=config
    )
    return response.text

# Test básico
print(get_completion("What is 1+1?"))

1 + 1 = 2


In [5]:
customer_email = """
Arrr, I be fuming that me blender lid flew off and splattered me kitchen walls 
with smoothie! And to make matters worse, the warranty don't cover the cost of 
cleaning up me kitchen. I need yer help right now, matey!
"""

style = "American English in a calm and respectful tone"

prompt = f"""Translate the text that is delimited by triple backticks \
into a style that is {style}.
text: ```{customer_email}```
"""

response_text = get_completion(prompt)
print("Traducción a estilo calmado:\n", response_text)

Traducción a estilo calmado:
 I am very disappointed that my blender lid came off and splattered smoothie all over my kitchen walls. To make matters worse, I discovered that the warranty does not cover the cost of cleaning up the mess. I would greatly appreciate your assistance with this matter as soon as possible.


In [6]:
from pydantic import BaseModel, Field

# 1. Definimos el esquema de datos tipado
class ReviewExtraction(BaseModel):
    gift: bool = Field(
        description="Was the item purchased as a gift for someone else? True if yes, False if not or unknown."
    )
    delivery_days: int = Field(
        description="How many days did it take for the product to arrive? If not found, output -1."
    )
    price_value: list[str] = Field(
        description="Sentences about the value or price as a list of strings."
    )

customer_review = """
This leaf blower is pretty amazing. It has four settings: candle blower, 
gentle breeze, windy city, and tornado. It arrived in two days, just in time 
for my wife's anniversary present. I think my wife liked it so much she was speechless. 
So far I've been the only one using it, and I've been using it every other morning 
to clear the leaves on our lawn. It's slightly more expensive than the other leaf blowers 
out there, but I think it's worth it for the extra features.
"""

# 2. Configuración con response_schema nativo (JSON validado)
config_estructurada = types.GenerateContentConfig(
    temperature=0.0,
    response_mime_type="application/json",
    response_schema=ReviewExtraction,
)

response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents=f"Extract the requested fields from this review:\n\n{customer_review}",
    config=config_estructurada
)

# 3. Acceso directo a los datos
print("Respuesta JSON bruta:")
print(response.text)

# Si tienes Pydantic instalado, puedes parsearlo a un objeto directamente:
datos_extraidos = ReviewExtraction.model_validate_json(response.text)
print("\nAcceso a campo tipado:")
print(f"¿Fue un regalo?: {datos_extraidos.gift}")
print(f"Días de entrega: {datos_extraidos.delivery_days}")
print(f"Frases de precio: {datos_extraidos.price_value}")

Respuesta JSON bruta:
{"gift":true,"delivery_days":2,"price_value":["It's slightly more expensive than the other leaf blowers out there, but I think it's worth it for the extra features."]}

Acceso a campo tipado:
¿Fue un regalo?: True
Días de entrega: 2
Frases de precio: ["It's slightly more expensive than the other leaf blowers out there, but I think it's worth it for the extra features."]


In [7]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

# 1. Initialize the model using LangChain with the exact model identifier
llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    temperature=0.0,
)

# 2. Define a clean, reusable prompt template using LCEL conventions
translation_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert communications assistant. Adapt the style while preserving the exact core meaning."),
    ("human", "Translate the text delimited by triple backticks into a style that is {style}.\n\ntext: ```{text}```")
])

# 3. Create the chain using the LCEL pipe operator (|)
translation_chain = translation_prompt | llm

# 4. Invoke the chain
customer_email = """
Arrr, I be fuming that me blender lid flew off and splattered me kitchen walls 
with smoothie! And to make matters worse, the warranty don't cover the cost of 
cleaning up me kitchen. I need yer help right now, matey!
"""

response = translation_chain.invoke({
    "style": "American English in a calm and respectful tone",
    "text": customer_email
})

print("--- LangChain Response ---")
print(response.content)

c:\Users\Usuario\Proyectos\proyectos-vscode\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


--- LangChain Response ---
[{'type': 'text', 'text': 'I am very frustrated that my blender lid came off and splattered smoothie all over my kitchen walls. To make matters worse, the warranty does not cover the cost of cleaning up the mess. I would greatly appreciate your immediate assistance with this issue.', 'extras': {'signature': 'Ep0bCpobARFNMg8sSbbx+qBmhJMZ3H2Dd124YsEbzMqK9TeSBy86MUHai+bQvHxypA0iLmZ5suMA6BmJ3JkKf5pZif42hJ8GpiQ36AxyQIj/wR3WoL2CIkjj4GKTHqvM0QVLgPaw1SknZfxuxOiZacEQQ+dGsiK0xo4T6Q7dzvowOs31WxmMsBkHiqj52blpl46OD3qQodMrzNuXkTacrFLpvSurI9KAOkCA0D7mdswOaoepEUVlYZ5aOE3hHeAIGG3JnshoUvhwg7f0DHSSx/T88Sjg2tB/JmEh/6WSKuwHQCAI5CkLWkzvGPmY+dibaXviVa1LivXoUGutn2Kyg8U6MG+Jqy6RijhxyD4TjjTuZowyGFMsxBJBT93cuHWUwFSNY+KNwpdeBbffPBkVRMyURDNNyszulVHxzvxVuj1GbvLBvIb+QyVVz8o7b0t1cseY3S5WkLz8LcMiKsbh2AC4B5QRMmoW5igoeUMfhImweEzeqIzRvoEzWYCkVnATrw0iejJKJEDLC6S/FqK/a9SWNq2QdAY5YSNz8g12gSb1AewraxrqYv8Pj8M5HQhIjQi3UudqFu9qod+G6Ijo5C8JS4GaaZdX7n2GSDiAEYazimmpJtLj9/a7lkv/RmTjKfwu8t0SZp6LhcE1qjSp8DI

In [8]:
from pydantic import BaseModel, Field

# 1. Define the target schema
class IncidentReport(BaseModel):
    summary: str = Field(description="A concise one-sentence summary of the core incident.")
    department: str = Field(description="Responsible department: Logistics, Operations, Customer Support, or Billing.")
    urgency_level: str = Field(description="Urgency classification: Low, Medium, or Critical.")
    action_items: list[str] = Field(description="List of concrete, actionable next steps required.")

# 2. Bind the schema directly to the model
structured_llm = llm.with_structured_output(IncidentReport)

# 3. Build the extraction chain
extraction_prompt = ChatPromptTemplate.from_messages([
    ("system", "Extract structured information from the provided raw input text."),
    ("human", "{raw_input}")
])

extraction_chain = extraction_prompt | structured_llm

# 4. Run inference
raw_audio_transcript = (
    "Hey team, quick update: the barcode scanner network in the central warehouse just went down "
    "and we have three trucks waiting to load. We need to restart the local server immediately "
    "and notify on-call infrastructure support because shipments are completely blocked."
)

report: IncidentReport = extraction_chain.invoke({"raw_input": raw_audio_transcript})

print("--- Structured Object Extracted ---")
print(f"Summary:     {report.summary}")
print(f"Department:  {report.department}")
print(f"Urgency:     {report.urgency_level}")
print(f"Action Items: {report.action_items}")

c:\Users\Usuario\Proyectos\proyectos-vscode\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


--- Structured Object Extracted ---
Summary:     The barcode scanner network in the central warehouse has failed, completely blocking shipments and leaving three trucks waiting.
Department:  Logistics
Urgency:     Critical
Action Items: ['Restart the local server immediately', 'Notify on-call infrastructure support']


In [9]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# 1. Initialize Gemini with temperature=0.0
llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    temperature=0.0,
)

# 2. Define prompt with dynamic history placeholder
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful and concise assistant."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

base_chain = chat_prompt | llm

# 3. Session storage dictionary (in production, use Redis or PostgreSQL)
session_store = {}

def get_session_history(session_id: str):
    if session_id not in session_store:
        session_store[session_id] = ChatMessageHistory()
    return session_store[session_id]

# 4. Wrap the LCEL chain with history management
conversational_chain = RunnableWithMessageHistory(
    base_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history"
)

# 5. Multi-turn test execution
config = {"configurable": {"session_id": "session_001"}}

res1 = conversational_chain.invoke({"input": "Hi, my name is Andrew."}, config=config)
print("Turn 1:", res1.content)

res2 = conversational_chain.invoke({"input": "What is 1 + 1?"}, config=config)
print("Turn 2:", res2.content)

res3 = conversational_chain.invoke({"input": "What is my name?"}, config=config)
print("Turn 3:", res3.content)

C:\Users\Usuario\AppData\Local\Temp\ipykernel_7228\2144716536.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.chat_message_histories import ChatMessageHistory
c:\Users\Usuario\Proyectos\proyectos-vscode\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)
c:\Users\Usuario\Proyectos\proyectos-vscode\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Turn 1: [{'type': 'text', 'text': 'Hello Andrew! How can I help you today?', 'extras': {'signature': 'Ep4ECpsEARFNMg+uE4V3z0zuqaJcHn8BiCqlfGdcj4u0wy6d18XJ/QKQuaB59wKNx7fZ0EJqAC33Ay+LAzqmf6nWOTfQrwzh6B/04pzu50NZ+GOl02+6GyyEJqnjdF0nxdu1Rfxc3EZJeoCOU36NlClmmjVQn9mM9Lq1+8LxgAOgGRJJ+DuJ3F45DwVtzcFo2XcN/XPeW4h6sS7Td8YOa7ckibazmZg5105tcIkCv98Petqmvl44O88lU0OWIu5wMWlEegt5fLR/n+jJ31ct+LG95jC04HZ6p7xRSBwBq6D+w5IeY7PfwuNB3VeuiUfmsx+dfo3JOmT/pFO1KRO+MJZK4fLdTA3oIFPgl6Y29jFojqdIb64Ui+eyW3dJAlujQTQZiYNiJOwIhiH8HqDyzG9ZQELr0SJPAwsk5KDBgUG0C5VRAFZ9fbQmaoqj1nBMysA1NXlIMjMSCLQwMDQugz1S1vvFRi9HsHWR8axPLcZuppCsHJook+jrpRGI1j+JJosJe2zWnk8P6+DHmCvaDOCkTZZrxaxzP9PA7EM/mrLfeaAy0p/ld8F3axVnv1oPCDzHPK4bkDbR15LkVjS0G2dVnOfAXeQOW0Y63ao2ZsFk43x3gS3/iPbyzxSddMzBdCXVGHHTt4JIIW2MyY8yzvYHyBCXJQ9pYOwyXM2aoQaCLg96jjPYbnezqJWXSAKDlFTfanruzNeJyrgQdfY5YSY='}}]


c:\Users\Usuario\Proyectos\proyectos-vscode\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Turn 2: [{'type': 'text', 'text': '1 + 1 = 2.', 'extras': {'signature': 'EvICCu8CARFNMg95FkAMPea3n55LU96BT9xrblAzFX3EH731UzrmQyfeZHpyPV6WfVlHztHv/Qu7gTWGtdaEdLDhkJAQkCZyxTFveCj+HmzkY9hWMg1/CgtINT62d6dgKopKJObOSFKJQIlWvdRSpOUxVdS32WX7+5OIbIvzJqvD1zXzKwL2jq7vDLms9QhjYxKhZnuy1VEMS+jUdDmFK05aNQiYCu84mhLh6Ukxl09/WeLnjM2tKCc/aoRsqldvYaN8Fv4zcW0bpDn1yQajxh9hlmSZs7ExcpEZbELHWbt2R4KBbPfxXnujy9dnRuZOmmfUWPQnZ3A/5vWNOiLHp6w5Imc+imA732tj8rQAQXKlyd7EPXeSLc/7nhrq9x01P55UwfbEBNBKgz8QhRBZwO/hnnRsa8tVfJVCmoeCz69qnjOw3FkWsYs7oYUIhIcSa5kAeLn/89tZabe5Kh651nh3prnq8cSUCUQvk5oz1bahFKptNw=='}}]


c:\Users\Usuario\Proyectos\proyectos-vscode\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Turn 3: [{'type': 'text', 'text': 'Your name is Andrew.', 'extras': {'signature': 'EvMCCvACARFNMg8Ekk1NKC7gRfXgg+G6bT02Z5Li3YgqN5RzZfi/403Dd0uFSxt8LdemksnBSwIUManjF+Kco7IfLvEBy1PYAXtYODobfLxID7tTRjHH58QhrWmasnuO8Au2eWf22zWEWrcPnvAY3t1fjIFVduG2uWhoN46XEMTyeJAYOB0aclooHzMZo0PYVlYNTSTyr/xJ2tW4xHxDJhD104vEN54/aedc4ZTd0GyfVvll2oKBM9/+1EMiojv/m8vJpIheQTvAhHNbjdOkvkPbFg+Uxa/7Zqrc6ayQzb0zzD3rYxjOUza+jbhh4b/hOf/vGNrVhazc/fN/00v1RVc0vTOhRh+zgYng/2Wndbdvr42EbJvnmVuzIc9t4S9+wrE+SRydQlDi5fzGMEghzf8spdFxRxl7u2HLdltI5SBB1AVHYzw8EPabdIGVdnb49kPedrgfLp4icBXQaXvlKdGT9Hxr2YRakNFDcFoHluSDhHymNQQ='}}]


In [10]:
from langchain_core.messages import HumanMessage, AIMessage, trim_messages

# 1. Simulated dialogue history
raw_history = [
    HumanMessage(content="Hi, my name is Andrew."),
    AIMessage(content="Hello Andrew, nice to meet you!"),
    HumanMessage(content="What is 1 + 1?"),
    AIMessage(content="1 + 1 equals 2."),
    HumanMessage(content="AI is what?"),
    AIMessage(content="AI is amazing!"),
    HumanMessage(content="Backpropagation is what?"),
    AIMessage(content="Backpropagation is beautiful!")
]

# 2. Prune history to retain only the last 4 messages (2 exchanges)
pruned_history = trim_messages(
    raw_history,
    strategy="last",
    token_counter=len,  # Counts message turns
    max_tokens=4,
    start_on="human",
    include_system=True
)

print("--- Trimmed Context Passed to Model ---")
for msg in pruned_history:
    print(f"[{msg.type}]: {msg.content}")

--- Trimmed Context Passed to Model ---
[human]: AI is what?
[ai]: AI is amazing!
[human]: Backpropagation is what?
[ai]: Backpropagation is beautiful!


In [11]:
summarizer_prompt = ChatPromptTemplate.from_messages([
    ("system", "Condense the provided schedule/meeting text into a concise, bulleted executive summary."),
    ("human", "{raw_schedule}")
])

summarizer_chain = summarizer_prompt | llm

schedule_text = """
There is a meeting at 8:00 AM with the product team regarding the Q3 roadmap.
From 9:00 AM to 12:00 PM, dedicate time to work on LangChain data pipelines.
At noon, lunch at the Italian restaurant with a customer who wants to understand the latest AI developments.
"""

summary_response = summarizer_chain.invoke({"raw_schedule": schedule_text})
print("--- Executive Summary ---")
print(summary_response.content)

c:\Users\Usuario\Proyectos\proyectos-vscode\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


--- Executive Summary ---
[{'type': 'text', 'text': '**Executive Summary:**\n\n* **8:00 AM:** Product team meeting regarding the Q3 roadmap.\n* **9:00 AM – 12:00 PM:** Work on LangChain data pipelines.\n* **12:00 PM:** Customer lunch at an Italian restaurant to discuss the latest AI developments.', 'extras': {'signature': 'Eq4LCqsLARFNMg9112Y5raNsqq5LKwgj6sIKs1/NWzcnG5fX2vXxo+gEw1mXH1ZT9b8vJvcxg3II1U9BMnCD/XyeoMcsBdSmnmReFKFIlZ1L1Kh5PAW6tv4dJdBGbr2rMnb1I4jkAf0sSdPZtf0Ip+Hu98rPbFOiVIfFKlV/Wm+YvupDTuzOTmvGB3oB4wINu1tK3+ED/lCrRwmaayvK2EpOjTFhvXn+mlKO3GD/QTitymV+VHmy/X6vnMLnrpr0pThKhAJhCeP7iAfexoCV5eJAyAIHHWlNm5admsdPBff4Hp5Q0JrCdfMEij38g+3aa5+OyirqBgLtAtcEj4atXsjV52uYTzn++gQX0sUXQjv+DZO5Eaotbrlmb61agmBVAQRFIRztOhcgBEaD0WpSk+1vRlODQhtJ4vctqR67ifbkb4bJTXLYUjGRFbYYqn5wdTmO0v+LECPHM7y4lCTdfdMkKwJ1wGTgU8S2s/lPbfhkBjWRMBtTTMemiUfbP5qSJknhyTiyoXP2XiWN/Ju8xfP5M/aaBz7QgudkiLYSplhkxJDnmrvrR2bH+1xAHYWZ6DE/o3Z7GaJnKY+yPQQ5cBDPCQHRNDKf4SmTFVheBIoPpEJiitHHHdJsGOA5uX2N6zIz6NvVIi1s336hADOJdSzlAM0jqhXnguO